# All models 尝试包含全部模型

包括:

* Rolling Normal；
* Linear Quantile；
* Fused K1；
* Fused K3；
* Structured K3；


# 3MDN Model: Structured Mixture Density Network 


本版采用以下数据流程：

1. 直接读取 `1DataDownload_Clean.ipynb` 生成的两张表，不重复生成 target、momentum 或 rolling features。
2. Rolling Normal 先使用原始 `micro_df` 计算，对预测月只使用之前 192 个日历月。
3. Micro 使用显式特征清单；`target_ret_final` 只作为标签，不进入特征清单。
4. Macro 在初始训练期拟合尺度，之后只使用每个时间块开始前的历史月份更新尺度。
5. 自动识别 local、Colab 和 Kaggle；`quick` 模式仅训练两个 epoch，并运行前两个 OOS 窗口。
6. Linear Quantile、Fused 和 Structured 模型在每个窗口重新初始化，不使用 warm start。

## Planned Robustness Checks

1. Normal versus Student-$t$ components.
2. $K=1$ versus multiple mixture components.
3. Rank versus normal-score micro normalization.
4. With and without the macro/FiLM channel.
5. Rolling versus expanding windows and uniform versus time-decay weights.

In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from datetime import datetime
import time

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

env = 'local'
# env = 'colab'
# env = 'kaggle'

if env == 'kaggle':
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(os.path.join(dirname, filename))

    micro_df = pd.read_parquet('/kaggle/input/mdndata-1990/micro_data.parquet')
    macro_df = pd.read_parquet('/kaggle/input/mdndata-1990/MacroFactor_McCracken.parquet')
    path = '/kaggle/working'

elif env == 'colab':
    import google.colab 
    # 1. 设置环境变量
    google.colab.drive.mount('/content/drive')
    os.environ.setdefault('KAGGLE_USERNAME', 'jianbinchenuc')
    if not os.getenv('KAGGLE_KEY'):
        raise RuntimeError("请先在运行环境中设置 KAGGLE_KEY。")

    # 2. 安装 kaggle 库 (如果尚未安装)
    !pip install -q kaggle

    # 3. 创建目录并下载数据集
    # -p 参数指定下载目录，--unzip 参数会自动解压
    !mkdir -p /content/mdndata
    !kaggle datasets download -d jianbinchenuc/mdndata-1990 -p /content/mdndata --unzip

    # 4. 验证文件是否已存在
    print("已下载的文件：", os.listdir('/content/mdndata'))
    micro_df = pd.read_parquet('/content/mdndata/micro_data.parquet') # 可以根据需要切换到 Colab 的路径
    macro_df = pd.read_parquet('/content/mdndata/MacroFactor_McCracken.parquet') # 可以根据需要切换到 Colab 的路径
    path = '/content/drive/MyDrive/Colab Notebooks/'

elif env == 'local':
    micro_df = pd.read_parquet('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/micro_data.parquet') # local 路径
    macro_df = pd.read_parquet('/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/data/MacroFactor_McCracken.parquet') # local 路径
    path = '/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/output'

else:
    raise ValueError("Invalid environment specified. Please choose 'kaggle', 'colab', or 'local'.")

timestamp = datetime.now().strftime("%Y%m%d_%H%M")


In [2]:
from pathlib import Path
import importlib.util
import copy
import json
import math
import os
import random
import sys
import warnings

import numpy as np
import pandas as pd
from scipy.stats import norm, t as student_t

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

# ==========================================
# 全局随机种子锁死函数
# ==========================================
def seed_everything(seed=42):
    """
    全局锁死随机种子，确保深度学习模型的绝对可复现性
    """
    # 1. 锁死 Python 内置随机库
    random.seed(seed)
    
    # 2. 锁死 Python 哈希表的随机化 (保证字典遍历顺序一致)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 3. 锁死 Numpy 随机种子
    np.random.seed(seed)
    
    # 4. 锁死 PyTorch CPU 随机种子
    torch.manual_seed(seed)
    
    # 5. 锁死 PyTorch GPU 随机种子
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # 如果你有双卡/多卡
        
        # 【极其关键】锁死 CuDNN 底层算法的随机性
        # 注意：设置为 True 可能会略微牺牲一点点 GPU 训练速度，但为了发论文复现，这是必须的！
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
    print(f"Global seed set to {seed} to ensure reproducibility.")

# ==========================================
# 0. 自动设备检测
# ==========================================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

random_seed = 42
seed_everything(random_seed)


Using device: mps
Global seed set to 42 to ensure reproducibility.


In [3]:
# ==========================================
# 1. 全局参数设置
# ==========================================
run_test = 'quick'  # 'quick' 或 'full'
random_seed = 42

initial_train_months = 180
val_months = 12
test_months = 12

# 每个窗口独立初始化，并使用 Validation early stopping。
early_stopping_max_epochs = 300
batch_size = 1024
lr = 1e-3
patience = 20
dropout_rate = 0.05
sigma_floor = 0.005  # 0.5% monthly component-scale floor

normalization_eps = 1e-8

windows_type = 'rolling'      # 'expanding' 或 'rolling'
weights_type = 'month_equal'  # 'uniform', 'month_equal', or 'time_decay'

ALPHAS = (0.01, 0.05, 0.10)
ALPHA_SUFFIX = {0.01: "1", 0.05: "5", 0.10: "10"}

QUANTILE_LEVELS = np.array([
    0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.10, 0.25, 0.50,
    0.75, 0.90, 0.95, 0.975, 0.99, 0.995, 0.9975, 0.999
], dtype=float)

In [4]:
print('Micro DataFrame shape:', micro_df.shape)
print('Micro months:', micro_df['mthcaldt'].nunique())
display(pd.concat([micro_df.head(5), micro_df.tail(5)]))
print("Macro DataFrame shape:", macro_df.shape)
display(pd.concat([macro_df.head(5), macro_df.tail(5)]))


Micro DataFrame shape: (1560360, 41)
Micro months: 408


,permno,mthcaldt,mthret,mthprc,mthcap,siccd,gvkey,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,de_ratio,cash_ratio,curr_ratio,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,ffi49,target_ret_final,MOM12m,mu_bench,sigma_bench,mthcap_log
0,10001,1991-01-31,0.013158,9.625,1.014475e+04,4920.0,012994,0.896385,5.690808,3.307560,0.445982,4.872598,14.070388,0.044401,0.099749,0.138304,0.082956,0.112235,0.112708,0.161306,0.392076,2.430626,0.061202,0.672101,1.796365,0.316439,1.340247,1.166318,8.789413,10.931698,3.261452,0.000000,0.0,0.0,-0.055775,31,0.012987,0.009425,0.002110,0.022682,9.224712
1,10001,1991-02-28,0.012987,9.750,1.027650e+04,4920.0,012994,0.975332,5.299653,3.037383,0.434947,5.205927,14.253121,0.047361,0.103356,0.141152,0.085157,0.121406,0.116860,0.168351,0.378677,2.249325,0.050190,0.614188,1.755852,0.289359,1.348320,1.192696,8.491285,10.618904,3.286892,0.000000,0.0,0.0,-0.042869,31,-0.012244,0.029179,0.003717,0.022716,9.237615
2,10001,1991-03-31,-0.012244,9.500,1.001300e+04,4920.0,012994,0.975332,5.299653,2.959502,0.423795,5.072442,13.887656,0.047361,0.103356,0.141152,0.085157,0.121406,0.116860,0.168351,0.378677,2.249325,0.050190,0.614188,1.755852,0.289359,1.348320,1.192696,8.491285,10.618904,3.286892,0.000000,0.0,0.0,-0.042869,31,0.039474,0.029348,0.001628,0.022954,9.211640
3,10001,1991-04-30,0.039474,9.875,1.040825e+04,4920.0,012994,0.975332,5.299653,3.115265,0.446100,5.339412,14.618585,0.047361,0.103356,0.141152,0.085157,0.121406,0.116860,0.168351,0.378677,2.249325,0.050190,0.614188,1.755852,0.289359,1.348320,1.192696,8.491285,10.618904,3.286892,0.000000,0.0,0.0,-0.042869,31,0.000000,0.016744,0.004917,0.025398,9.250354
4,10001,1991-05-31,0.000000,9.875,1.040825e+04,4920.0,012994,1.040348,5.316239,3.086420,0.444182,6.893394,14.618585,0.047832,0.103923,0.139492,0.083400,0.119963,0.115336,0.165372,0.375721,2.271979,0.059539,0.472059,1.708091,0.318617,1.389858,1.185531,8.331080,10.836074,3.210526,0.000000,0.0,0.0,-0.019640,31,0.079241,0.070429,0.005972,0.024859,9.250354
1560355,93436,2024-08-31,-0.077390,214.110,6.840044e+08,3711.0,184996,0.105166,57.713252,59.310249,7.174588,59.301712,335.706250,0.131885,0.075809,0.229558,0.183071,0.217032,0.200551,0.207100,0.095088,0.822338,0.293684,0.409772,0.696792,1.017029,1.760274,0.902169,26.475384,5.220111,1.543949,0.047001,0.0,0.0,0.010123,23,0.221942,-0.100783,-0.006184,0.139169,20.343475
1560356,93436,2024-09-30,0.221942,261.630,8.390474e+08,3711.0,184996,0.105166,57.713252,72.473684,8.802612,72.758187,346.216280,0.131885,0.075809,0.229558,0.183071,0.217032,0.200551,0.207100,0.095088,0.822338,0.293684,0.409772,0.696792,1.017029,1.760274,0.902169,26.475384,5.220111,1.543949,0.047001,0.0,0.0,0.010123,23,-0.045025,-0.144312,0.014849,0.153502,20.547778
1560357,93436,2024-10-31,-0.045025,249.850,8.020335e+08,3711.0,184996,0.105166,57.713252,69.210526,8.406271,69.482219,369.690039,0.131885,0.075809,0.229558,0.183071,0.217032,0.200551,0.207100,0.095088,0.822338,0.293684,0.409772,0.696792,1.017029,1.760274,0.902169,26.475384,5.220111,1.543949,0.047001,0.0,0.0,0.010123,23,0.381469,0.302679,0.027542,0.140071,20.502661
1560358,93436,2024-11-30,0.381469,345.160,1.107984e+09,3711.0,184996,0.083346,53.762150,93.286486,11.394010,76.450592,306.628288,0.132692,0.084756,0.234822,0.183922,0.208738,0.190668,0.203448,0.099842,0.839369,0.274887,0.497103,0.690047,1.047903,1.798235,0.866390,25.589359,5.285830,1.475261,0.044858,0.0,0.0,-0.014637,23,0.170008,0.040696,0.043050,0.167882,20.825808
1560359,93436,2024-12-31,0.170008,403.840,1.298958e+09,3711.0,184996,0.083346,53.762150,109.145946,13.368497,89.698836,274.076290,0.132692,0.084756,0.234822,0.183922,0.208738,0.190668,0.203448,0.099842,0.839369,0.274887,0.497103,0.690047,1.047903,1.798235,0.866390,25.589359,5.285830,1.475261,0.044858,0.0,0.0,-0.014637,23,0.001882,0.389088,0.054301,0.171772,20.984828


Macro DataFrame shape: (408, 20)


,mthcaldt,PAYEMS,UNRATE,CLAIMSx,INDPRO,DPCERA3M086SBEA,AMDMNOx,ISRATIOx,HOUST,M2SL,NONBORRES,BUSLOANS,FEDFUNDS,CP3Mx,UEMPMEAN,CPIAUCSL,VIXCLSx,BAAFFM,TERM_SPREAD,CREDIT_SPREAD
0,1991-01-31,-0.001000,0.1,0.023836,-0.005920,-0.000778,0.040338,0.02,6.876265,0.001402,-0.069067,0.000930,-0.50,-0.11,0.1,0.001485,22.5845,3.12,1.34,2.35
1,1991-02-28,-0.001892,0.1,-0.041536,-0.004931,-0.003955,-0.029746,0.02,6.682109,0.001999,-0.093898,-0.007471,-0.40,-0.70,-0.3,-0.000014,26.9318,3.54,1.87,2.36
2,1991-03-31,-0.002384,0.2,0.091651,-0.007933,-0.009491,0.005330,0.02,6.872128,0.000704,0.100378,0.005295,-0.66,-0.61,0.5,-0.002975,22.1126,3.82,1.91,2.22
3,1991-04-30,-0.001568,0.2,0.024146,-0.009247,0.006162,-0.061953,0.00,6.825460,-0.000088,0.043726,-0.002343,-0.13,-0.08,0.2,-0.000742,19.0410,3.97,2.20,1.98
4,1991-05-31,-0.001709,-0.1,-0.082863,0.003497,0.011971,0.041812,0.00,6.908755,-0.002301,0.034535,-0.002986,-0.21,-0.34,0.6,0.002221,18.9272,4.03,2.39,1.90
403,2024-08-31,0.000718,0.2,0.006105,-0.006401,0.003767,0.094333,0.00,7.121252,-0.002100,-0.024829,-0.006629,0.00,0.03,-0.1,0.002110,14.4084,0.51,-0.95,1.59
404,2024-09-30,0.000895,-0.1,-0.030903,0.008110,0.001463,0.000459,-0.01,7.212294,0.003887,0.030865,0.003711,0.00,-0.19,0.4,0.000322,19.6750,0.27,-1.18,1.73
405,2024-10-31,0.001598,-0.1,-0.029656,-0.002832,0.003543,-0.007867,0.01,7.210818,-0.001050,-0.031354,0.002598,-0.20,-0.26,1.6,-0.000073,17.6597,0.29,-1.00,1.70
406,2024-11-30,0.000075,0.0,0.055299,-0.002629,0.001225,0.002425,0.00,7.178545,0.000319,0.023649,-0.000157,-0.30,-0.24,0.3,0.000641,19.9478,0.80,-0.41,1.53
407,2024-12-31,0.001426,0.1,-0.078988,-0.001472,0.002754,-0.010575,0.00,7.161622,0.002139,0.024533,-0.002535,-0.19,-0.09,0.8,0.000686,15.9822,1.13,-0.06,1.42


### ==========================================
### Rolling Normal：使用预测月之前 192 个日历月
### ==========================================

In [ ]:
ROLLING_SCALE_FLOOR = 1e-6


def normal_var_and_loss_es(alpha, pred_mean, pred_vol):
    """计算 Normal distribution 的左尾 VaR 和 loss-ES。"""
    z_alpha = norm.ppf(alpha)
    var_value = pred_mean + pred_vol * z_alpha
    es_return = pred_mean - pred_vol * norm.pdf(z_alpha) / alpha
    return float(var_value), float(-es_return)


def run_rolling_normal(panel, max_history_months):
    """
    对每只股票使用预测月之前 max_history_months 个日历月的收益率估计 Normal distribution。
    历史窗口不包含预测月；股票缺月时不会用更早的观测补足 192 条。
    """
    panel = panel.sort_values(["permno", "mthcaldt"]).reset_index(drop=True).copy()
    panel["mthcaldt"] = pd.to_datetime(panel["mthcaldt"]) + pd.offsets.MonthEnd(0)

    # 将日期转为连续月份编号，便于严格按日历月确定每只股票的左边界。
    month_id = panel["mthcaldt"].dt.year.to_numpy() * 12 + panel["mthcaldt"].dt.month.to_numpy()
    returns = panel["mthret"].to_numpy(dtype=float)
    pred_mean = np.full(len(panel), np.nan)
    pred_vol = np.full(len(panel), np.nan)
    history_n = np.zeros(len(panel), dtype=np.int32)

    for positions in panel.groupby("permno", sort=False).indices.values():
        positions = np.asarray(positions)
        stock_months = month_id[positions]
        stock_returns = returns[positions]

        # 对月份 t，只使用 [t-max_history_months, t-1] 的收益率。
        left = np.searchsorted(stock_months, stock_months - max_history_months, side="left")
        right = np.arange(len(positions))  # 排除当月
        count = right - left

        cumulative_sum = np.r_[0.0, np.cumsum(stock_returns)]
        cumulative_sq_sum = np.r_[0.0, np.cumsum(stock_returns ** 2)]
        window_sum = cumulative_sum[right] - cumulative_sum[left]
        window_sq_sum = cumulative_sq_sum[right] - cumulative_sq_sum[left]

        has_mean = count > 0
        has_vol = count > 1
        pred_mean[positions[has_mean]] = window_sum[has_mean] / count[has_mean]
        variance = (window_sq_sum[has_vol] - window_sum[has_vol] ** 2 / count[has_vol]) / (count[has_vol] - 1)
        pred_vol[positions[has_vol]] = np.sqrt(np.maximum(variance, 0.0))
        history_n[positions] = count

    panel["pred_mean"] = pred_mean
    panel["pred_vol"] = pred_vol
    panel["history_n"] = history_n

    # 新上市股票历史不足时，使用上一个月的横截面统计，不使用预测月信息。
    monthly_fallback = panel.groupby("mthcaldt")["mthret"].agg(["mean", "std"]).shift(1)
    panel["pred_mean"] = panel["pred_mean"].fillna(panel["mthcaldt"].map(monthly_fallback["mean"]))
    panel["pred_vol"] = panel["pred_vol"].fillna(panel["mthcaldt"].map(monthly_fallback["std"]))
    panel["pred_vol"] = panel["pred_vol"].clip(lower=ROLLING_SCALE_FLOOR)

    available_dates = pd.DatetimeIndex(np.sort(panel["mthcaldt"].unique()))
    oos_panel = panel[panel["mthcaldt"] >= available_dates[max_history_months]].copy()
    if run_test == "quick":
        quick_dates = np.sort(oos_panel["mthcaldt"].unique())[:2 * test_months]
        oos_panel = oos_panel[oos_panel["mthcaldt"].isin(quick_dates)].copy()

    output_rows = []
    for _, row in tqdm(oos_panel.iterrows(), total=len(oos_panel), desc="Rolling Normal"):
        output_row = {
            "permno": row["permno"], "mthcaldt": row["mthcaldt"],
            "target_ret_final": float(row["target_ret_final"]), "model_id": "rolling_normal",
            "pred_mean": float(row["pred_mean"]), "pred_vol": float(row["pred_vol"]),
            "history_n": int(row["history_n"]),
        }
        for alpha in ALPHAS:
            suffix = ALPHA_SUFFIX[alpha]
            var_value, es_loss = normal_var_and_loss_es(alpha, output_row["pred_mean"], output_row["pred_vol"])
            output_row[f"var_{suffix}"] = var_value
            output_row[f"es_{suffix}"] = es_loss
        output_rows.append(output_row)

    return pd.DataFrame(output_rows).sort_values(["mthcaldt", "permno"]).reset_index(drop=True)


rolling_normal_forecasts = run_rolling_normal(
    panel=micro_df, max_history_months=initial_train_months + val_months
)

In [ ]:
rolling_normal_forecasts.to_parquet(
    os.path.join(path, f'rolling_normal_forecasts_{timestamp}.parquet'), index=False
)

display(pd.concat([rolling_normal_forecasts.head(4), rolling_normal_forecasts.tail(4)]))

rolling_normal_summary = rolling_normal_forecasts.groupby("mthcaldt").agg(
    n_firms=("permno", "size"), mean_history_n=("history_n", "mean"),
    mean_predicted_vol=("pred_vol", "mean"), mean_es_5=("es_5", "mean"),
)
display(rolling_normal_summary.head())

In [5]:
# =========================================================
# 1. Validate the cleaned micro table and normalize explicit features
# =========================================================
micro_normalize = 'rank'  # 'normal' or 'rank' for accounting-ratio features

micro_df = micro_df.copy()

micro_exclude_cols = {
    'permno', 'mthcaldt', 'mthcap', 'mthprc', 'siccd', 'gvkey', 'ffi49', 'target_ret_final'
}

micro_cols = [
    col for col in micro_df.columns
    if col not in micro_exclude_cols
]

micro_df = micro_df[['permno', 'mthcaldt', 'ffi49', 'target_ret_final'] + micro_cols].copy()
micro_df['mthcaldt'] = pd.to_datetime(micro_df['mthcaldt']) + pd.offsets.MonthEnd(0)

if micro_normalize == 'rank':
    micro_norm = (
        micro_df.groupby('mthcaldt')[micro_cols].rank(pct=True) * 2.0 - 1.0
    )
elif micro_normalize == 'normal':
    monthly_mean = micro_df.groupby('mthcaldt')[micro_cols].transform('mean')
    monthly_std = micro_df.groupby('mthcaldt')[micro_cols].transform('std')
    micro_norm = (
        micro_df[micro_cols] - monthly_mean
    ) / (monthly_std + normalization_eps)
else:
    raise ValueError("micro_normalize must be 'normal' or 'rank'")
micro_df[micro_cols] = micro_norm.fillna(0.0).to_numpy()

micro_df = micro_df.sort_values(['mthcaldt', 'permno']).reset_index(drop=True)

print('Micro model panel:', micro_df.shape)

display(pd.concat([micro_df.head(3), micro_df.tail(3)]))

Micro model panel: (1560360, 37)


,permno,mthcaldt,ffi49,target_ret_final,mthret,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,de_ratio,cash_ratio,curr_ratio,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,MOM12m,mu_bench,sigma_bench,mthcap_log
0,10001,1991-01-31,31,0.012987,-0.328160,0.218879,-0.536902,-0.756098,-0.483054,-0.351283,0.527399,-0.256889,0.173899,-0.796009,-0.334812,0.159328,0.288565,-0.477352,0.663605,0.424137,-0.615458,0.554007,0.405765,0.273361,-0.590751,0.244853,0.569845,0.356984,0.514729,-0.292683,-0.281596,-0.168514,-0.401330,0.264492,-0.057966,-0.986696,-0.899905
1,10009,1991-01-31,0,0.038462,-0.586950,0.005702,-0.009186,0.000950,-0.004118,0.002217,0.032309,0.000317,-0.000950,-0.004118,-0.029458,-0.000634,0.008236,0.001584,0.002217,0.000317,-0.025974,0.035793,0.000317,-0.020272,0.038961,0.002534,0.004118,0.012987,0.002534,-0.292683,-0.281596,-0.168514,-0.002217,0.450111,0.172632,-0.559709,-0.961989
2,10010,1991-01-31,12,0.058824,0.854926,-0.882167,0.863161,0.937282,0.832753,0.958822,0.947418,0.275895,0.175166,0.735192,-0.380424,0.454545,0.683244,0.877732,-0.482420,-0.530567,0.683244,-0.536902,-0.608489,-0.318340,0.574913,0.712385,-0.457080,0.641432,0.228381,0.707317,-0.281596,-0.168514,0.730757,0.998733,0.999366,0.997466,-0.119417
1560357,93426,2024-12-31,37,-0.007243,0.721739,0.657739,-0.152696,0.424000,-0.563130,-0.094261,0.214609,-0.135304,-0.105391,0.113739,-0.149217,-0.259826,-0.040696,0.449043,-0.352348,-0.302261,0.661913,-0.088696,-0.609043,0.523478,0.627130,0.260522,0.102609,0.626435,-0.162783,0.556174,-0.448696,-0.208348,0.070609,-0.858087,-0.835826,-0.236870,-0.672348
1560358,93434,2024-12-31,1,0.147685,0.901913,0.995130,-0.805913,-0.713391,-0.841391,-0.707130,-0.549217,-0.819130,-0.791304,-0.583304,-0.873391,-0.842783,-0.868522,-0.402435,0.414261,-0.938087,0.954087,-0.746087,0.094957,-0.986087,-0.779478,-0.300174,-0.639652,-0.659826,-0.273043,0.592348,-0.448696,-0.208348,-0.811478,-0.923478,0.825391,0.975652,-0.988870
1560359,93436,2024-12-31,23,0.001882,0.918609,-0.859478,0.885913,0.924174,0.871304,0.941565,0.987130,0.499130,-0.148522,-0.588870,0.440696,0.682783,0.807304,0.071304,-0.421913,-0.414261,0.554783,-0.001739,-0.419130,0.495652,-0.076174,0.460174,0.831652,-0.286957,0.200696,0.512348,-0.448696,-0.208348,0.433739,0.451826,0.764870,0.641739,0.997217


In [6]:
# =========================================================
# 2. Macro：初始训练期统一拟合，之后按时间块更新
# =========================================================
def standardize_macro(
    macro_data,
    macro_cols,
    model_dates,
    initial_months,
    update_months,
):
    macro = macro_data.set_index('mthcaldt').loc[model_dates, macro_cols].copy()
    macro_z = pd.DataFrame(index=model_dates, columns=macro_cols, dtype=float)

    initial_dates = model_dates[:initial_months]
    initial_mean = macro.loc[initial_dates].mean()
    initial_std = macro.loc[initial_dates].std(ddof=0)
    macro_z.loc[initial_dates] = (
        (macro.loc[initial_dates] - initial_mean) /
        (initial_std + normalization_eps)
    ).to_numpy()

    for start in range(initial_months, len(model_dates), update_months):
        # print(f'>>> Standardizing Macro features for block starting at {model_dates[start]}...')
        block_dates = model_dates[start:start + update_months]
        history = macro.iloc[:start]
        history_mean = history.mean()
        history_std = history.std(ddof=0)
        macro_z.loc[block_dates] = (
            (macro.loc[block_dates] - history_mean) /
            (history_std + normalization_eps)
        ).to_numpy()

    macro_z.index.name = 'mthcaldt'
    return macro_z.reset_index()

macro_cols = [
    col for col in macro_df.columns
    if col != 'mthcaldt'
]

macro_df['mthcaldt'] = pd.to_datetime(macro_df['mthcaldt']) + pd.offsets.MonthEnd(0)
macro_df = macro_df.sort_values('mthcaldt').reset_index(drop=True)

model_dates = pd.DatetimeIndex(np.sort(micro_df['mthcaldt'].unique()))

macro_df = standardize_macro(
    macro_data=macro_df,
    macro_cols=macro_cols,
    model_dates=model_dates,
    initial_months=initial_train_months,
    update_months=test_months,
)
pd.concat([macro_df.head(5), macro_df.tail(5)])

,mthcaldt,PAYEMS,UNRATE,CLAIMSx,INDPRO,DPCERA3M086SBEA,AMDMNOx,ISRATIOx,HOUST,M2SL,NONBORRES,BUSLOANS,FEDFUNDS,CP3Mx,UEMPMEAN,CPIAUCSL,VIXCLSx,BAAFFM,TERM_SPREAD,CREDIT_SPREAD
0,1991-01-31,-1.544357,0.729992,0.539483,-1.669163,-0.863438,1.088412,1.775543,-2.377725,0.388363,-1.063207,0.140443,-2.459411,-0.427037,0.124777,0.601691,0.322572,-0.473714,-0.532523,0.452030
1,1991-02-28,-2.192501,0.729992,-0.817965,-1.469064,-1.649264,-1.007572,1.775543,-3.404600,0.550763,-1.445587,-1.227084,-1.945786,-3.240398,-0.599735,0.013918,0.924305,-0.184668,-0.058573,0.470548
2,1991-03-31,-2.549766,1.417942,1.947638,-2.076241,-3.018345,0.041438,1.775543,-2.399603,0.198073,1.546106,0.850978,-3.281213,-2.811241,0.849289,-1.146876,0.257253,0.008029,-0.022803,0.211304
3,1991-04-30,-1.957019,1.417942,0.545911,-2.342198,0.852805,-1.970789,0.173549,-2.646426,-0.017617,0.673709,-0.392313,-0.558996,-0.283985,0.305905,-0.271394,-0.167903,0.111260,0.236528,-0.233113
4,1991-05-31,-2.059530,-0.645909,-1.676114,0.236066,2.289584,1.132493,0.173549,-2.205888,-0.620199,0.532180,-0.496941,-0.969897,-1.523771,1.030417,0.890301,-0.183655,0.152552,0.406434,-0.381253
403,2024-08-31,-0.019744,0.360853,0.054717,-0.858590,0.157267,2.364269,0.067085,-0.036957,-0.404763,-0.034278,-0.618382,0.033549,0.170575,-0.106644,0.687854,-0.684488,-2.068622,-2.216100,-1.064560
404,2024-09-30,0.002850,-0.163222,-0.189031,0.681921,-0.066753,-0.049609,-0.415929,0.225697,0.730580,0.042472,0.346484,0.033549,-0.915719,0.336030,0.108108,-0.033047,-2.216690,-2.410739,-0.868202
405,2024-10-31,0.092972,-0.163222,-0.180820,-0.479714,0.135471,-0.263698,0.550100,0.221439,-0.205664,-0.043270,0.242691,-1.037861,-1.261358,1.398448,-0.020236,-0.282325,-2.204351,-2.258413,-0.910279
406,2024-11-30,-0.102179,0.011470,0.378732,-0.458106,-0.089945,0.000934,0.067085,0.128332,0.053851,0.032528,-0.014406,-1.573566,-1.162604,0.247495,0.211272,0.000696,-1.889707,-1.759122,-1.148713
407,2024-12-31,0.070973,0.186161,-0.505745,-0.335332,0.058751,-0.333345,0.067085,0.079508,0.399053,0.033746,-0.236290,-0.984291,-0.421949,0.690169,0.226152,-0.489820,-1.686113,-1.462933,-1.302994


In [7]:
# =========================================================
# 3. 最后合并标准化后的 Micro 和 Macro
# =========================================================
processed_df = (
    micro_df
    .merge(macro_df, on='mthcaldt', how='left', validate='many_to_one')
    .sort_values(['mthcaldt', 'permno'])
    .reset_index(drop=True)
)

print(f'Data preprocessing complete. Final shape: {processed_df.shape}')
print(f'Micro features: {len(micro_cols)}; Macro features: {len(macro_cols)}')

pd.concat([processed_df.head(5), processed_df.tail(5)])

Data preprocessing complete. Final shape: (1560360, 56)
Micro features: 33; Macro features: 19


,permno,mthcaldt,ffi49,target_ret_final,mthret,bm,evm,pe_exi,ps,pcf,pfcf,npm,opmad,gpm,cfm,roe,aftret_invcapx,gprof,debt_at,debt_ebitda,short_debt,ocf_lct,de_ratio,cash_ratio,curr_ratio,at_turn,rect_turn,pay_turn,sale_equity,rd_sale,adv_sale,staff_sale,accrual,MOM12m,mu_bench,sigma_bench,mthcap_log,PAYEMS,UNRATE,CLAIMSx,INDPRO,DPCERA3M086SBEA,AMDMNOx,ISRATIOx,HOUST,M2SL,NONBORRES,BUSLOANS,FEDFUNDS,CP3Mx,UEMPMEAN,CPIAUCSL,VIXCLSx,BAAFFM,TERM_SPREAD,CREDIT_SPREAD
0,10001,1991-01-31,31,0.012987,-0.328160,0.218879,-0.536902,-0.756098,-0.483054,-0.351283,0.527399,-0.256889,0.173899,-0.796009,-0.334812,0.159328,0.288565,-0.477352,0.663605,0.424137,-0.615458,0.554007,0.405765,0.273361,-0.590751,0.244853,0.569845,0.356984,0.514729,-0.292683,-0.281596,-0.168514,-0.401330,0.264492,-0.057966,-0.986696,-0.899905,-1.544357,0.729992,0.539483,-1.669163,-0.863438,1.088412,1.775543,-2.377725,0.388363,-1.063207,0.140443,-2.459411,-0.427037,0.124777,0.601691,0.322572,-0.473714,-0.532523,0.452030
1,10009,1991-01-31,0,0.038462,-0.586950,0.005702,-0.009186,0.000950,-0.004118,0.002217,0.032309,0.000317,-0.000950,-0.004118,-0.029458,-0.000634,0.008236,0.001584,0.002217,0.000317,-0.025974,0.035793,0.000317,-0.020272,0.038961,0.002534,0.004118,0.012987,0.002534,-0.292683,-0.281596,-0.168514,-0.002217,0.450111,0.172632,-0.559709,-0.961989,-1.544357,0.729992,0.539483,-1.669163,-0.863438,1.088412,1.775543,-2.377725,0.388363,-1.063207,0.140443,-2.459411,-0.427037,0.124777,0.601691,0.322572,-0.473714,-0.532523,0.452030
2,10010,1991-01-31,12,0.058824,0.854926,-0.882167,0.863161,0.937282,0.832753,0.958822,0.947418,0.275895,0.175166,0.735192,-0.380424,0.454545,0.683244,0.877732,-0.482420,-0.530567,0.683244,-0.536902,-0.608489,-0.318340,0.574913,0.712385,-0.457080,0.641432,0.228381,0.707317,-0.281596,-0.168514,0.730757,0.998733,0.999366,0.997466,-0.119417,-1.544357,0.729992,0.539483,-1.669163,-0.863438,1.088412,1.775543,-2.377725,0.388363,-1.063207,0.140443,-2.459411,-0.427037,0.124777,0.601691,0.322572,-0.473714,-0.532523,0.452030
3,10016,1991-01-31,34,0.161039,-0.695280,0.594552,-0.727589,-0.723155,-0.853659,-0.960089,-0.486855,-0.371555,-0.181501,-0.333545,-0.486221,0.375356,0.482420,0.291099,0.669940,0.422236,-0.335445,-0.361419,0.528033,0.321508,-0.173266,0.365220,-0.312639,0.501425,0.683877,0.552106,-0.281596,-0.168514,0.820082,-0.766867,-0.865695,0.092176,0.185303,-1.544357,0.729992,0.539483,-1.669163,-0.863438,1.088412,1.775543,-2.377725,0.388363,-1.063207,0.140443,-2.459411,-0.427037,0.124777,0.601691,0.322572,-0.473714,-0.532523,0.452030
4,10019,1991-01-31,38,0.076923,-0.436490,0.272094,-0.408299,-0.658537,-0.187203,-0.337346,0.242952,0.785873,0.632563,0.500792,0.547038,0.741527,-0.789357,0.645233,-0.551473,-0.642065,-0.696547,0.901806,-0.809946,0.814381,0.913209,0.413367,-0.290466,0.915109,-0.427304,0.863161,-0.281596,-0.168514,0.371555,-0.870130,-0.909408,0.255622,-0.519797,-1.544357,0.729992,0.539483,-1.669163,-0.863438,1.088412,1.775543,-2.377725,0.388363,-1.063207,0.140443,-2.459411,-0.427037,0.124777,0.601691,0.322572,-0.473714,-0.532523,0.452030
1560355,93374,2024-12-31,46,0.012492,-0.394783,0.446261,0.550609,0.870609,-0.490783,0.152696,0.385739,-0.332870,-0.411478,-0.879652,-0.389217,-0.362783,-0.074087,-0.725217,-0.261913,0.594435,0.619478,-0.571478,0.316174,-0.132522,-0.400348,-0.233391,-0.330783,0.168000,-0.013565,-0.384696,-0.448696,-0.208348,0.250783,-0.138087,-0.307826,-0.483130,0.446957,0.070973,0.186161,-0.505745,-0.335332,0.058751,-0.333345,0.067085,0.079508,0.399053,0.033746,-0.236290,-0.984291,-0.421949,0.690169,0.226152,-0.489820,-1.686113,-1.462933,-1.302994
1560356,93397,2024-12-31,1,-0.057236,-0.536696,-0.051826,0.965913,0.844174,0.072696,0.650087,0.812174,-0.146435,-0.641739,-0.757217,-0.211826,-0.276522,-0.093565,-0.327304,-0.288000,0.873391,-0.273391,0.259130,-0.549913,-0.842783,-0.891478,0.186783,0.586087,0.252870,-0.149913,-0.384696,-0.448696,-0.208348,0.161043,0.399652,0.166609,-0.481043,-0.535304,0.070973,0.186161,-0.5057

### ==========================================
### Linear Quantile：线性分位数回归
### ==========================================

In [10]:
linear_batch_size = 8192
linear_lr = 1e-3
linear_weight_decay = 1e-4

In [11]:
def tensor_batch_indices(length, batch_size, device, shuffle=True):
    if shuffle:
        indices = torch.randperm(length, device=device)
    else:
        indices = torch.arange(length, device=device)
    for start in range(0, length, batch_size):
        yield indices[start:start + batch_size] 

In [12]:
# ==========================================
# 2. 轻量级时间窗口生成器
# ==========================================
class DataWindowGenerator:
    """
    仅负责按时间切分数据，向模型输送 Train/Val/Test 窗口
    """
    def __init__(self, df, date_col='mthcaldt'):
        self.df = df
        self.date_col = date_col

    def expanding_window_generator(self, initial_train_months=180, val_months=12, test_months=12):
        dates = np.sort(self.df[self.date_col].unique())
        
        start_idx = 0
        current_split_idx = initial_train_months
        
        while current_split_idx + val_months  < len(dates):
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            train_data = self.df[self.df[self.date_col].isin(train_dates)]
            val_data = self.df[self.df[self.date_col].isin(val_dates)]
            test_data = self.df[self.df[self.date_col].isin(test_dates)]
            
            window_info = {
                'train': (pd.Timestamp(train_dates[0]).strftime('%Y-%m'), pd.Timestamp(train_dates[-1]).strftime('%Y-%m')),
                'val': (pd.Timestamp(val_dates[0]).strftime('%Y-%m'), pd.Timestamp(val_dates[-1]).strftime('%Y-%m')),
                'test': (pd.Timestamp(test_dates[0]).strftime('%Y-%m'), pd.Timestamp(test_dates[-1]).strftime('%Y-%m'))
            }
            yield window_info, train_data, val_data, test_data
            current_split_idx += test_months
            
    # （固定 15 年滚动窗口版）
    def rolling_window_generator(self, train_months=180, val_months=12, test_months=12):
        """
        固定训练窗口长度的滚动窗口生成器。
        参数：
        train_months: 固定训练集包含的月数。默认 180 个月(15年)。
        """
        # 确保日期是有序的
        dates = np.sort(self.df[self.date_col].unique())
        
        # 初始时，当前切分点就是训练集的总长度（180个月）
        current_split_idx = train_months
        
        # 只要后面还有足够的月份可以分给验证集和测试集，就继续滚动
        while current_split_idx + val_months < len(dates):
            
            # 【核心修改点】让训练集的起点跟着一起滚动，确保长度永远等于 train_months
            start_idx = current_split_idx - train_months
            
            # 切分对应的日期数组
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            # 根据日期过滤出对应的数据集
            train_data = self.df[self.df[self.date_col].isin(train_dates)]
            val_data = self.df[self.df[self.date_col].isin(val_dates)]
            test_data = self.df[self.df[self.date_col].isin(test_dates)]
            
            window_info = {
                'train': (pd.Timestamp(train_dates[0]).strftime('%Y-%m'), pd.Timestamp(train_dates[-1]).strftime('%Y-%m')),
                'val': (pd.Timestamp(val_dates[0]).strftime('%Y-%m'), pd.Timestamp(val_dates[-1]).strftime('%Y-%m')),
                'test': (pd.Timestamp(test_dates[0]).strftime('%Y-%m'), pd.Timestamp(test_dates[-1]).strftime('%Y-%m'))
            }
            yield window_info, train_data, val_data, test_data
            
            # 向前推進一个测试窗口的长度
            current_split_idx += test_months

def make_window_generator(panel):
    generator = DataWindowGenerator(df=panel, date_col="mthcaldt")

    if windows_type == "expanding":
        return generator.expanding_window_generator(
            initial_train_months=initial_train_months,
            val_months=val_months,
            test_months=test_months,
        )

    return generator.rolling_window_generator(
        train_months=initial_train_months,
        val_months=val_months,
        test_months=test_months,
    )

In [13]:
# ==========================================
# Linear Quantile 模型
# Macro + Micro + FF49 行业固定效应
# ==========================================
class LinearQuantile(nn.Module):
    def __init__(self, macro_dim, micro_dim, num_industries, quantile_levels):
        super().__init__()
        self.num_industries = num_industries
        input_dim = macro_dim + micro_dim + num_industries - 1
        self.linear = nn.Linear(input_dim, len(quantile_levels))

        quantiles = torch.tensor(quantile_levels, dtype=torch.float32).view(1, -1)
        self.register_buffer("quantile_levels", quantiles)

    def forward(self, macro, micro, industry):
        industry_dummy = F.one_hot(
            industry.long(), num_classes=self.num_industries
        ).to(macro.dtype)[:, 1:]

        x = torch.cat([macro, micro, industry_dummy], dim=1)
        return self.linear(x)

In [14]:
def weighted_pinball_loss(pred, target, weight, quantile_levels):
    target = target.reshape(-1, 1)
    weight = weight.reshape(-1)

    error = target - pred
    loss = torch.maximum(
        quantile_levels * error,
        (quantile_levels - 1.0) * error,
    )
    loss_per_stock = loss.mean(dim=1)

    return (
        (loss_per_stock * weight).sum()
        / weight.sum().clamp_min(1e-12)
    )

In [15]:
# ==========================================
# Sample weights: prevent months with more listed stocks from dominating
# ==========================================
def calculate_sample_weights(df_dates, mode='month_equal', half_life_months=36):
    dates = pd.to_datetime(df_dates).reset_index(drop=True)
    if mode == 'uniform':
        weights = np.ones(len(dates), dtype=np.float64)
    elif mode in {'month_equal', 'time_decay'}:
        month_counts = dates.groupby(dates).transform('size').to_numpy(dtype=float)
        weights = 1.0 / np.maximum(month_counts, 1.0)
        if mode == 'time_decay':
            max_date = dates.max()
            months_diff = (
                (max_date.year - dates.dt.year) * 12
                + (max_date.month - dates.dt.month)
            ).to_numpy(dtype=float)
            weights *= np.power(0.5, months_diff / half_life_months)
    else:
        raise ValueError("weights_type must be 'uniform', 'month_equal', or 'time_decay'")
    weights = weights / weights.mean()
    return weights.astype(np.float32)

# ==========================================
# 将完整面板转成设备端 Tensor
# ==========================================
def make_device_tensors(panel):
    X_macro = torch.from_numpy(
        panel[macro_cols].to_numpy(dtype=np.float32, copy=True)
    ).to(device)

    X_micro = torch.from_numpy(
        panel[micro_cols].to_numpy(dtype=np.float32, copy=True)
    ).to(device)

    industry = torch.from_numpy(
        panel["ffi49"].to_numpy(dtype=np.int64, copy=True)
    ).to(device)

    target = torch.from_numpy(
        panel["target_ret_final"].to_numpy(dtype=np.float32, copy=True)
    ).to(device)

    return X_macro, X_micro, industry, target

In [16]:
# ==========================================
# Linear Quantile 训练
# ==========================================
def train_linear_quantile(
    model,
    X_train_macro, X_train_micro, X_train_industry, y_train, w_train,
    X_val_macro, X_val_micro, X_val_industry, y_val, w_val,
    epochs=300, batch_size=8192, lr=1e-3, patience=20,
):
    if run_test == "quick":
        epochs = 2
        patience = 1

    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=linear_weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.8,
        patience=8,
        min_lr=1e-6,
    )

    train_loss_history = []
    val_loss_history = []
    best_val_loss = float("inf")
    best_model_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    train_size = len(y_train)

    for epoch in range(epochs):
        model.train()
        train_loss_numerator = 0.0
        train_weight_sum = 0.0

        for batch_idx in tensor_batch_indices(
            train_size,
            batch_size,
            X_train_macro.device,
            shuffle=True,
        ):
            b_macro = X_train_macro[batch_idx]
            b_micro = X_train_micro[batch_idx]
            b_industry = X_train_industry[batch_idx]
            b_y = y_train[batch_idx]
            b_w = w_train[batch_idx]

            optimizer.zero_grad()
            pred = model(b_macro, b_micro, b_industry)
            loss = weighted_pinball_loss(
                pred,
                b_y,
                b_w,
                model.quantile_levels,
            )
            loss.backward()
            optimizer.step()

            batch_weight = float(b_w.sum().item())
            train_loss_numerator += loss.item() * batch_weight
            train_weight_sum += batch_weight

        train_loss = train_loss_numerator / max(train_weight_sum, 1e-12)
        train_loss_history.append(train_loss)

        model.eval()
        with torch.no_grad():
            val_pred = model(
                X_val_macro,
                X_val_micro,
                X_val_industry,
            )
            val_loss = weighted_pinball_loss(
                val_pred,
                y_val,
                w_val,
                model.quantile_levels,
            ).item()

        if not np.isfinite(val_loss):
            raise RuntimeError("Linear Quantile validation loss became non-finite.")

        val_loss_history.append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    model.load_state_dict(best_model_state)

    return (
        model,
        best_val_loss,
        train_loss_history,
        val_loss_history,
    )

In [17]:
# ==========================================
# 从预测分位数计算均值、波动率、VaR 和 ES
# ==========================================
def add_quantile_distribution_results(output, quantile_pred):
    levels = QUANTILE_LEVELS
    quantile_pred = np.sort(quantile_pred, axis=1)
    trapz = np.trapezoid

    full_levels = np.r_[0.0, levels, 1.0]
    full_quantiles = np.column_stack([
        quantile_pred[:, 0],
        quantile_pred,
        quantile_pred[:, -1],
    ])

    pred_mean = trapz(full_quantiles, full_levels, axis=1)
    second_moment = trapz(full_quantiles ** 2, full_levels, axis=1)

    output["pred_mean"] = pred_mean
    output["pred_vol"] = np.sqrt(
        np.maximum(second_moment - pred_mean ** 2, 0.0)
    )

    for alpha in ALPHAS:
        suffix = ALPHA_SUFFIX[alpha]
        alpha_idx = np.flatnonzero(np.isclose(levels, alpha))[0]
        below_alpha = levels < alpha

        var_value = quantile_pred[:, alpha_idx]
        tail_levels = np.r_[0.0, levels[below_alpha], alpha]
        tail_quantiles = np.column_stack([
            quantile_pred[:, 0],
            quantile_pred[:, below_alpha],
            var_value,
        ])

        output[f"var_{suffix}"] = var_value
        output[f"es_{suffix}"] = (
            -trapz(tail_quantiles, tail_levels, axis=1) / alpha
        )

    output["quantile_levels"] = json.dumps(levels.tolist())
    output["quantile_vec"] = [
        json.dumps(row.tolist()) for row in quantile_pred
    ]

    return output

In [18]:
# ==========================================
# Linear Quantile 滚动样本外预测
# ==========================================
def run_linear_quantile(panel):
    panel = panel.sort_values(["mthcaldt", "permno"]).reset_index(drop=True)
    window_gen = make_window_generator(panel)

    global_X_mac, global_X_mic, global_ind, global_Y = make_device_tensors(panel)
    
    all_predictions = []
    history_logs = []

    for window_idx, (info, train_df, val_df, test_df) in enumerate(window_gen):
        print(
            f"[Window {window_idx}] "
            f"Train: {info['train'][0]}~{info['train'][1]} | "
            f"Val: {info['val'][0]}~{info['val'][1]} | "
            f"Test: {info['test'][0]}~{info['test'][1]}"
        )

        start_time = time.time()
        # 提取各集合在全局大表中的行索引。
        idx_tr = train_df.index.values
        idx_va = val_df.index.values
        idx_te = test_df.index.values

        # 直接切片预先计算好的标准化值。
        X_tr_mac = global_X_mac[idx_tr]
        X_va_mac = global_X_mac[idx_va]
        X_te_mac = global_X_mac[idx_te]

        X_tr_mic, X_tr_ind, Y_tr = global_X_mic[idx_tr], global_ind[idx_tr], global_Y[idx_tr]
        X_va_mic, X_va_ind, Y_va = global_X_mic[idx_va], global_ind[idx_va], global_Y[idx_va]
        X_te_mic, X_te_ind       = global_X_mic[idx_te], global_ind[idx_te]


        w_tr = torch.from_numpy(
            calculate_sample_weights(
                train_df["mthcaldt"],
                mode=weights_type,
                half_life_months=36,
            )
        ).to(device)

        w_va = torch.from_numpy(
            calculate_sample_weights(
                val_df["mthcaldt"],
                mode=weights_type,
                half_life_months=36,
            )
        ).to(device)

        # 每个窗口重新初始化，不使用 warm start
        seed_everything(random_seed + window_idx)

        model = LinearQuantile(
            macro_dim=len(macro_cols),
            micro_dim=len(micro_cols),
            num_industries=50,
            quantile_levels=QUANTILE_LEVELS,
        ).to(device)

        # 使用训练收益率的无条件分位数初始化截距
        initial_quantiles = np.quantile(
            train_df["target_ret_final"].to_numpy(),
            QUANTILE_LEVELS,
        )

        with torch.no_grad():
            model.linear.weight.zero_()
            model.linear.bias.copy_(
                torch.tensor(
                    initial_quantiles,
                    dtype=torch.float32,
                    device=device,
                )
            )

        model, best_val_loss, tr_hist, va_hist = train_linear_quantile(
            model=model,
            X_train_macro=X_tr_mac,
            X_train_micro=X_tr_mic,
            X_train_industry=X_tr_ind,
            y_train=Y_tr,
            w_train=w_tr,
            X_val_macro=X_va_mac,
            X_val_micro=X_va_mic,
            X_val_industry=X_va_ind,
            y_val=Y_va,
            w_val=w_va,
            epochs=early_stopping_max_epochs,
            batch_size=linear_batch_size,
            lr=linear_lr,
            patience=patience,
        )

        model.eval()
        with torch.no_grad():
            quantile_pred = model(
                X_te_mac,
                X_te_mic,
                X_te_ind,
            ).cpu().numpy()

        output = test_df[
            ["permno", "mthcaldt", "target_ret_final"]
        ].copy()

        output["model_id"] = "linear_quantile"
        output["validation_loss"] = best_val_loss
        output["epochs_run"] = len(tr_hist)

        output = add_quantile_distribution_results(
            output,
            quantile_pred,
        )
        all_predictions.append(output)

        history_logs.append({
            "window": window_idx,
            "train_period": f"{info['train'][0]} to {info['train'][1]}",
            "test_period": f"{info['test'][0]} to {info['test'][1]}",
            "duration_sec": time.time() - start_time,
            "best_val_loss": best_val_loss,
            "epochs_run": len(tr_hist),
            "train_loss_trace": tr_hist,
            "val_loss_trace": va_hist,
        })

        if run_test == "quick" and window_idx == 1:
            break

    forecasts = pd.concat(all_predictions, ignore_index=True)
    metrics = pd.DataFrame(history_logs)

    return forecasts, metrics

In [19]:
linear_quantile_forecasts, linear_quantile_metrics = run_linear_quantile(processed_df)
# 执行保存
linear_quantile_forecasts.to_parquet(os.path.join(path, f'linear_quantile_forecasts_{timestamp}.parquet'), index=False)
linear_quantile_metrics.to_parquet(os.path.join(path, f'linear_quantile_metrics_{timestamp}.parquet'), index=False)

display(pd.concat([linear_quantile_forecasts.head(4),linear_quantile_forecasts.tail(4),]))

[Window 0] Train: 1991-01~2005-12 | Val: 2006-01~2006-12 | Test: 2007-01~2007-12
Global seed set to 42 to ensure reproducibility.
[Window 1] Train: 1992-01~2006-12 | Val: 2007-01~2007-12 | Test: 2008-01~2008-12
Global seed set to 43 to ensure reproducibility.


,permno,mthcaldt,target_ret_final,model_id,validation_loss,epochs_run,pred_mean,pred_vol,var_1,es_1,var_5,es_5,var_10,es_10,quantile_levels,quantile_vec
0,10001,2007-01-31,0.264774,linear_quantile,0.010741,2,0.004042,0.148063,-0.308416,0.378099,-0.188881,0.265611,-0.141851,0.215488,"[0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1,...","[-0.48360776901245117, -0.406444251537323, -0...."
1,10002,2007-01-31,0.033054,linear_quantile,0.010741,2,0.017135,0.123476,-0.278229,0.343799,-0.164725,0.235077,-0.122589,0.189367,"[0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1,...","[-0.4297482967376709, -0.3727389872074127, -0...."
2,10025,2007-01-31,-0.028094,linear_quantile,0.010741,2,0.017027,0.179373,-0.395330,0.483404,-0.242747,0.341349,-0.179266,0.276178,"[0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1,...","[-0.6227901577949524, -0.52052241563797, -0.45..."
3,10026,2007-01-31,-0.038517,linear_quantile,0.010741,2,0.013380,0.122415,-0.222720,0.273598,-0.159270,0.204338,-0.114400,0.170587,"[0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1,...","[-0.35051923990249634, -0.2959788739681244, -0..."
87371,92869,2008-12-31,-0.366006,linear_quantile,0.012190,2,0.020446,0.295905,-0.792829,0.984607,-0.434802,0.629851,-0.274519,0.492256,"[0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1,...","[-1.2418242692947388, -1.0828405618667603, -0...."
87372,92872,2008-12-31,-0.175000,linear_quantile,0.012190,2,0.029607,0.278757,-0.741796,0.932268,-0.419158,0.592736,-0.266125,0.467689,"[0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1,...","[-1.1924731731414795, -1.0149005651474, -0.893..."
87373,92874,2008-12-31,-0.496259,linear_quantile,0.012190,2,0.046866,0.237663,-0.625193,0.780696,-0.329855,0.486260,-0.191353,0.373432,"[0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1,...","[-1.0146946907043457, -0.8601034283638, -0.732..."
87374,93105,2008-12-31,0.106070,linear_quantile,0.012190,2,0.040474,0.269384,-0.674337,0.838224,-0.387221,0.539613,-0.241218,0.426916,"[0.001, 0.0025, 0.005, 0.01, 0.025, 0.05, 0.1,...","[-1.066042184829712, -0.9236394166946411, -0.7..."


### ==========================================
### Fused models (K=1 and 3)
### ==========================================

In [20]:
# ==========================================
# Fused MDN
# K=1：Conditional Normal
# K=3：Fused Normal Mixture
# ==========================================
class FusedMDN(nn.Module):
    def __init__(self, macro_dim, micro_dim, num_industries, ind_emb_dim=6,
        hidden_dim=64, head_dim=8, num_components=1, dropout_rate=0.05, dist_type="normal",
    ):
        super().__init__()

        self.num_components = num_components
        self.dist_type = dist_type
        self.ind_embedding = nn.Embedding(num_embeddings=num_industries, embedding_dim=ind_emb_dim)
        input_dim = macro_dim + micro_dim + ind_emb_dim

        self.fused_extractor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ELU(),
            nn.Dropout(dropout_rate),

            nn.Linear(hidden_dim // 2, head_dim),
            nn.LayerNorm(head_dim),
            nn.ELU(),
        )

        self.pi_head = nn.Linear(head_dim, num_components) if num_components > 1 else None
        self.mu_head = nn.Linear(head_dim, num_components)
        self.sigma_head = nn.Linear(head_dim, num_components)
        self.nu_head = nn.Linear(head_dim, num_components) if dist_type == "t" else None

        # 行业 embedding 不要初始化得太大
        nn.init.normal_(self.ind_embedding.weight, mean=0.0, std=0.05)

        nn.init.xavier_normal_(self.mu_head.weight, gain=0.1)
        nn.init.zeros_(self.mu_head.bias)

        nn.init.xavier_normal_(self.sigma_head.weight, gain=0.1)
        nn.init.constant_(self.sigma_head.bias, -2.35)

        if self.pi_head is not None:
            # K=3 初始概率相同
            nn.init.zeros_(self.pi_head.weight)
            nn.init.zeros_(self.pi_head.bias)

        if self.nu_head is not None:
            nn.init.xavier_normal_(self.nu_head.weight, gain=0.1)
            nn.init.constant_(self.nu_head.bias, 5.0)

    def forward(self, x_macro, x_micro, x_industry):
        industry_embedding = self.ind_embedding(x_industry)

        x = torch.cat([x_macro, x_micro, industry_embedding], dim=1,)
        h = self.fused_extractor(x)

        if self.pi_head is None:
            pi_logits = h.new_zeros((len(h), 1))
        else:
            pi_logits = self.pi_head(h)

        mu = self.mu_head(h)
        sigma = F.softplus(self.sigma_head(h)) + sigma_floor

        if self.nu_head is None:
            nu = None
        else:
            nu = F.softplus(self.nu_head(h)) + 2.01

        return pi_logits, mu, sigma, nu

In [21]:
# ==========================================
# 所有 MDN 共用的加权 NLL
# ==========================================
def mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, target, weights, dist_type="normal",):
    target = target.reshape(-1, 1)
    weights = weights.reshape(-1)

    log_pi = torch.log_softmax(pi_logits, dim=1)

    if dist_type == "normal":
        dist = torch.distributions.Normal(mu, sigma)
    elif dist_type == "t":
        dist = torch.distributions.StudentT(df=nu, loc=mu, scale=sigma,)
    else:
        raise ValueError("dist_type must be 'normal' or 't'")

    log_component = dist.log_prob(target.expand_as(mu))
    observation_nll = -torch.logsumexp( log_pi + log_component, dim=1,)

    return (observation_nll * weights).sum() / weights.sum().clamp_min(1e-12)

In [22]:
# ==========================================
# 所有 MDN 共用的 Validation
# ==========================================
def evaluate_mdn(model, X_macro, X_micro, X_industry, target, weights, batch_size,):
    model.eval()
    loss_numerator = 0.0
    weight_sum = 0.0

    with torch.no_grad():
        for batch_idx in tensor_batch_indices(len(target), batch_size, X_macro.device, shuffle=False):
            b_macro = X_macro[batch_idx]
            b_micro = X_micro[batch_idx]
            b_industry = X_industry[batch_idx]
            b_y = target[batch_idx]
            b_w = weights[batch_idx]

            pi_logits, mu, sigma, nu = model(b_macro, b_micro, b_industry)

            loss = mdn_nll_loss_weighted(pi_logits, mu, sigma, nu, b_y, b_w, dist_type=model.dist_type)

            batch_weight = float(b_w.sum().item())
            loss_numerator += loss.item() * batch_weight
            weight_sum += batch_weight

    return loss_numerator / max(weight_sum, 1e-12)

In [23]:
# ==========================================
# 所有 MDN 共用的 Early-stopping 训练
# ==========================================
def train_mdn(
    model, X_train_macro, X_train_micro, X_train_industry, y_train, w_train, X_val_macro, 
    X_val_micro, X_val_industry, y_val, w_val, epochs=300, batch_size=1024, lr=1e-3, patience=20,
):
    if run_test == "quick":
        epochs = 2
        patience = 1

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4,)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.8, patience=8, min_lr=1e-6,)

    best_val_loss = float("inf")
    best_model_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0

    train_loss_history = []
    val_loss_history = []

    for epoch in range(epochs):
        model.train()
        train_loss_numerator = 0.0
        train_weight_sum = 0.0

        for batch_idx in tensor_batch_indices(
            len(y_train), batch_size, X_train_macro.device, shuffle=True,
        ):
            b_macro = X_train_macro[batch_idx]
            b_micro = X_train_micro[batch_idx]
            b_industry = X_train_industry[batch_idx]
            b_y = y_train[batch_idx]
            b_w = w_train[batch_idx]

            optimizer.zero_grad()

            pi_logits, mu, sigma, nu = model(b_macro, b_micro,b_industry,)

            loss = mdn_nll_loss_weighted(
                pi_logits, mu, sigma, nu, b_y, b_w, dist_type=model.dist_type,
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0,)
            optimizer.step()

            batch_weight = float(b_w.sum().item())
            train_loss_numerator += loss.item() * batch_weight
            train_weight_sum += batch_weight

        train_loss = train_loss_numerator / max(train_weight_sum, 1e-12)
        train_loss_history.append(train_loss)

        val_loss = evaluate_mdn(
            model, X_val_macro, X_val_micro, X_val_industry, y_val, w_val, batch_size,
        )
        if not np.isfinite(val_loss):
            raise RuntimeError("Validation NLL became non-finite.")

        val_loss_history.append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

            if epochs_no_improve >= patience:
                break

    model.load_state_dict(best_model_state)

    return model, best_val_loss, train_loss_history, val_loss_history

In [24]:
# ==========================================
# 所有 MDN 共用的滚动样本外预测
# ==========================================
def run_mdn_model(panel, model_class, model_name, num_components, 
                  dist_type="normal", model_kwargs=None,
):
    
    panel = panel.sort_values(["mthcaldt", "permno"]).reset_index(drop=True)

    model_kwargs = {} if model_kwargs is None else model_kwargs
    window_gen = make_window_generator(panel)
    global_X_mac, global_X_mic, global_ind, global_y = make_device_tensors(panel)

    all_predictions = []
    history_logs = []
    embedding_history = []

    for window_idx, (info, train_df, val_df, test_df) in enumerate(window_gen):
        print(
            f"[{model_name} K={num_components} | Window {window_idx}] "
            f"Train: {info['train'][0]}~{info['train'][1]} | "
            f"Val: {info['val'][0]}~{info['val'][1]} | "
            f"Test: {info['test'][0]}~{info['test'][1]}"
        )

        start_time = time.time()

        # 提取各集合在全局大表中的行索引。
        idx_tr = train_df.index.values
        idx_va = val_df.index.values
        idx_te = test_df.index.values

        # 直接切片预先计算好的标准化值。
        X_tr_mac = global_X_mac[idx_tr]
        X_va_mac = global_X_mac[idx_va]
        X_te_mac = global_X_mac[idx_te]

        X_tr_mic, X_tr_ind, y_tr = global_X_mic[idx_tr], global_ind[idx_tr], global_y[idx_tr]
        X_va_mic, X_va_ind, y_va = global_X_mic[idx_va], global_ind[idx_va], global_y[idx_va]
        X_te_mic, X_te_ind       = global_X_mic[idx_te], global_ind[idx_te]

        w_tr = torch.from_numpy(
            calculate_sample_weights(train_df["mthcaldt"], mode=weights_type, half_life_months=36)
        ).to(device)

        w_va = torch.from_numpy(
            calculate_sample_weights(val_df["mthcaldt"], mode=weights_type, half_life_months=36)
        ).to(device)

        # 每个窗口独立初始化，不使用 warm start
        seed_everything(random_seed + window_idx)

        model = model_class(macro_dim=len(macro_cols), micro_dim=len(micro_cols), num_industries=50,
            num_components=num_components, dropout_rate=dropout_rate, dist_type=dist_type, **model_kwargs,
        ).to(device)

        model, best_val_loss, tr_hist, va_hist = train_mdn(
            model=model, X_train_macro=X_tr_mac, X_train_micro=X_tr_mic, X_train_industry=X_tr_ind,
            y_train=y_tr, w_train=w_tr, X_val_macro=X_va_mac, X_val_micro=X_va_mic, X_val_industry=X_va_ind,
            y_val=y_va, w_val=w_va, epochs=early_stopping_max_epochs, batch_size=batch_size, lr=lr,
            patience=patience)

        model.eval()

        with torch.no_grad():
            pi_logits, mu, sigma, nu = model(X_te_mac, X_te_mic, X_te_ind)
            pi = torch.softmax(pi_logits, dim=1)

        output = test_df[["permno", "mthcaldt", "target_ret_final"]].copy()

        output["model_id"] = f"{model_name}_{dist_type}_k{num_components}"
        output["pi_vec"] = [json.dumps(row.tolist()) for row in pi.cpu().numpy()]
        output["pi_logits_vec"] = [json.dumps(row.tolist()) for row in pi_logits.cpu().numpy()]
        output["mu_vec"] = [json.dumps(row.tolist()) for row in mu.cpu().numpy()]
        output["sigma_vec"] = [json.dumps(row.tolist()) for row in sigma.cpu().numpy()]
        output["nu_vec"] = [json.dumps(row.tolist()) for row in nu.cpu().numpy()] if nu is not None else None

        all_predictions.append(output)

        embedding_weights = model.ind_embedding.weight.detach().cpu().numpy()
        window_embeddings = pd.DataFrame(
            embedding_weights, columns=[f"emb_dim_{i + 1}" for i in range(embedding_weights.shape[1])]
        )
        window_embeddings["ffi49"] = window_embeddings.index
        window_embeddings["window"] = window_idx
        window_embeddings["test_period"] = f"{info['test'][0]} to {info['test'][1]}"
        embedding_history.append(window_embeddings)

        history_logs.append({
            "window": window_idx,
            "model_id": f"{model_name}_{dist_type}_k{num_components}",
            "train_period": f"{info['train'][0]} to {info['train'][1]}",
            "test_period": f"{info['test'][0]} to {info['test'][1]}",
            "duration_sec": time.time() - start_time,
            "best_val_loss": best_val_loss,
            "epochs_run": len(tr_hist),
            "train_loss_trace": tr_hist,
            "val_loss_trace": va_hist,
        })

        if run_test == "quick" and window_idx == 1:
            break

    forecasts = pd.concat(all_predictions, ignore_index=True)
    metrics = pd.DataFrame(history_logs)
    embeddings = pd.concat(embedding_history, ignore_index=True)
    return forecasts, metrics, embeddings

In [25]:
fused_k1_forecasts, fused_k1_metrics, fused_k1_embeddings = run_mdn_model(
    panel=processed_df,
    model_class=FusedMDN,
    model_name="fused",
    num_components=1,
    dist_type="normal",
    model_kwargs={"ind_emb_dim": 6, "hidden_dim": 64, "head_dim": 8},
)
# 执行保存
fused_k1_forecasts.to_parquet(os.path.join(path, f'fused_k1_forecasts_{timestamp}.parquet'), index=False)
fused_k1_metrics.to_parquet(os.path.join(path, f'fused_k1_metrics_{timestamp}.parquet'), index=False)
fused_k1_embeddings.to_parquet(os.path.join(path, f'fused_k1_embeddings_{timestamp}.parquet'), index=False)


display(pd.concat([fused_k1_forecasts.head(4), fused_k1_forecasts.tail(4),]))
display(pd.concat([fused_k1_metrics.head(2), fused_k1_metrics.tail(2),]))
display(pd.concat([fused_k1_embeddings.head(4), fused_k1_embeddings.tail(4),]))

[fused K=1 | Window 0] Train: 1991-01~2005-12 | Val: 2006-01~2006-12 | Test: 2007-01~2007-12
Global seed set to 42 to ensure reproducibility.
[fused K=1 | Window 1] Train: 1992-01~2006-12 | Val: 2007-01~2007-12 | Test: 2008-01~2008-12
Global seed set to 43 to ensure reproducibility.


,permno,mthcaldt,target_ret_final,model_id,pi_vec,pi_logits_vec,mu_vec,sigma_vec,nu_vec
0,10001,2007-01-31,0.264774,fused_normal_k1,[1.0],[0.0],[-0.023840896785259247],[0.11002255231142044],None
1,10002,2007-01-31,0.033054,fused_normal_k1,[1.0],[0.0],[0.008810965344309807],[0.08598149567842484],None
2,10025,2007-01-31,-0.028094,fused_normal_k1,[1.0],[0.0],[0.0009863171726465225],[0.1531534641981125],None
3,10026,2007-01-31,-0.038517,fused_normal_k1,[1.0],[0.0],[0.0011718161404132843],[0.08343377709388733],None
87371,92869,2008-12-31,-0.366006,fused_normal_k1,[1.0],[0.0],[-0.019180644303560257],[0.1273656189441681],None
87372,92872,2008-12-31,-0.175000,fused_normal_k1,[1.0],[0.0],[-0.0064608193933963776],[0.14683617651462555],None
87373,92874,2008-12-31,-0.496259,fused_normal_k1,[1.0],[0.0],[-0.0012707039713859558],[0.10919301956892014],None
87374,93105,2008-12-31,0.106070,fused_normal_k1,[1.0],[0.0],[-0.010686513036489487],[0.1419159173965454],None


,window,model_id,train_period,test_period,duration_sec,best_val_loss,epochs_run,train_loss_trace,val_loss_trace
0,0,fused_normal_k1,1991-01 to 2005-12,2007-01 to 2007-12,21.518756,-0.935107,2,"[-0.700661863628424, -0.7499042743261624]","[-0.9351074764435082, -0.9078448219516901]"
1,1,fused_normal_k1,1992-01 to 2006-12,2008-01 to 2008-12,20.093754,-0.814188,2,"[-0.7211895331600997, -0.763766773580604]","[-0.8098656384556118, -0.8141879783480451]"
0,0,fused_normal_k1,1991-01 to 2005-12,2007-01 to 2007-12,21.518756,-0.935107,2,"[-0.700661863628424, -0.7499042743261624]","[-0.9351074764435082, -0.9078448219516901]"
1,1,fused_normal_k1,1992-01 to 2006-12,2008-01 to 2008-12,20.093754,-0.814188,2,"[-0.7211895331600997, -0.763766773580604]","[-0.8098656384556118, -0.8141879783480451]"


,emb_dim_1,emb_dim_2,emb_dim_3,emb_dim_4,emb_dim_5,emb_dim_6,ffi49
0,0.108248,0.054741,-0.020171,-0.002249,-0.063944,0.172786,0
1,0.064093,0.091108,-0.163274,0.118531,-0.123483,0.291614,1
2,0.085912,0.098615,-0.011785,0.046694,-0.172371,0.131679,2
3,0.062066,0.106384,-0.040061,0.015211,-0.034313,-0.048918,3
46,-0.063988,0.012923,-0.018605,-0.033637,0.066032,0.073665,46
47,0.171312,0.121300,-0.358823,0.084243,-0.020413,0.314651,47
48,0.111300,0.020235,0.101475,-0.021031,-0.013711,-0.071944,48
49,-0.099345,-0.081917,-0.133956,-0.066724,0.102753,-0.051221,49


In [26]:
fused_k3_forecasts, fused_k3_metrics, fused_k3_embeddings = run_mdn_model(
    processed_df,
    FusedMDN,
    "fused",
    num_components=3,
    dist_type="normal",
    model_kwargs={"ind_emb_dim": 6, "hidden_dim": 64, "head_dim": 8},
)

# 执行保存
fused_k3_forecasts.to_parquet(os.path.join(path, f'fused_k3_forecasts_{timestamp}.parquet'), index=False)
fused_k3_metrics.to_parquet(os.path.join(path, f'fused_k3_metrics_{timestamp}.parquet'), index=False)
fused_k3_embeddings.to_parquet(os.path.join(path, f'fused_k3_embeddings_{timestamp}.parquet'), index=False)

display(pd.concat([fused_k3_forecasts.head(4), fused_k3_forecasts.tail(4),]))
display(pd.concat([fused_k3_metrics.head(2), fused_k3_metrics.tail(2),]))
display(pd.concat([fused_k3_embeddings.head(4), fused_k3_embeddings.tail(4),]))

[fused K=3 | Window 0] Train: 1991-01~2005-12 | Val: 2006-01~2006-12 | Test: 2007-01~2007-12
Global seed set to 42 to ensure reproducibility.
[fused K=3 | Window 1] Train: 1992-01~2006-12 | Val: 2007-01~2007-12 | Test: 2008-01~2008-12
Global seed set to 43 to ensure reproducibility.


,permno,mthcaldt,target_ret_final,model_id,pi_vec,pi_logits_vec,mu_vec,sigma_vec,nu_vec
0,10001,2007-01-31,0.264774,fused_normal_k3,"[0.0408945232629776, 0.31203243136405945, 0.64...","[-1.617624044418335, 0.41448691487312317, 1.14...","[0.13766013085842133, -0.026875626295804977, -...","[0.24507278203964233, 0.11181851476430893, 0.0...",None
1,10002,2007-01-31,0.033054,fused_normal_k3,"[0.04155033081769943, 0.30030637979507446, 0.6...","[-1.6142789125442505, 0.36361899971961975, 1.1...","[0.11272987723350525, 2.786889672279358e-05, -...","[0.23092716932296753, 0.10776543617248535, 0.0...",None
2,10025,2007-01-31,-0.028094,fused_normal_k3,"[0.051207125186920166, 0.4476114809513092, 0.5...","[-1.4449247121810913, 0.7231221199035645, 0.83...","[0.09254123270511627, 0.007141582667827606, -0...","[0.3594036400318146, 0.15878722071647644, 0.08...",None
3,10026,2007-01-31,-0.038517,fused_normal_k3,"[0.037154775112867355, 0.28237321972846985, 0....","[-1.6742210388183594, 0.35391634702682495, 1.2...","[0.08294235914945602, 0.008284259587526321, -0...","[0.23684920370578766, 0.10578381270170212, 0.0...",None
87371,92869,2008-12-31,-0.366006,fused_normal_k3,"[0.34415528178215027, 0.5040400624275208, 0.15...","[0.05679662525653839, 0.4383593797683716, -0.7...","[0.11231088638305664, 0.03446605056524277, 0.0...","[0.13927039504051208, 0.10034141689538956, 0.2...",None
87372,92872,2008-12-31,-0.175000,fused_normal_k3,"[0.38325488567352295, 0.45833221077919006, 0.1...","[0.1716909259557724, 0.3505849242210388, -0.71...","[0.1067056953907013, 0.02979462780058384, 0.07...","[0.15177321434020996, 0.11323229968547821, 0.2...",None
87373,92874,2008-12-31,-0.496259,fused_normal_k3,"[0.2798711955547333, 0.6035313606262207, 0.116...","[-0.10028751194477081, 0.6681809425354004, -0....","[0.11086347699165344, 0.032716698944568634, 0....","[0.1302187740802765, 0.08536422997713089, 0.23...",None
87374,93105,2008-12-31,0.106070,fused_normal_k3,"[0.38251110911369324, 0.4525259733200073, 0.16...","[0.1632644087076187, 0.3313518464565277, -0.67...","[0.11502744257450104, 0.03223329782485962, 0.0...","[0.149797260761261, 0.1129741221666336, 0.2882...",None


,window,model_id,train_period,test_period,duration_sec,best_val_loss,epochs_run,train_loss_trace,val_loss_trace
0,0,fused_normal_k3,1991-01 to 2005-12,2007-01 to 2007-12,22.666772,-0.977277,2,"[-0.7625664398128041, -0.8264130375275148]","[-0.9772765141003564, -0.9465620023903794]"
1,1,fused_normal_k3,1992-01 to 2006-12,2008-01 to 2008-12,23.698238,-0.896852,2,"[-0.7774337748672794, -0.8382298095911687]","[-0.8968521004162845, -0.8859597014795945]"
0,0,fused_normal_k3,1991-01 to 2005-12,2007-01 to 2007-12,22.666772,-0.977277,2,"[-0.7625664398128041, -0.8264130375275148]","[-0.9772765141003564, -0.9465620023903794]"
1,1,fused_normal_k3,1992-01 to 2006-12,2008-01 to 2008-12,23.698238,-0.896852,2,"[-0.7774337748672794, -0.8382298095911687]","[-0.8968521004162845, -0.8859597014795945]"


,emb_dim_1,emb_dim_2,emb_dim_3,emb_dim_4,emb_dim_5,emb_dim_6,ffi49
0,-0.067987,-0.068672,0.039435,-0.144975,0.084025,0.170143,0
1,-0.187266,-0.060686,-0.049888,-0.053794,0.157090,0.115576,1
2,-0.120274,-0.187900,-0.182457,-0.075855,0.114030,0.090026,2
3,-0.071188,-0.146463,0.010545,-0.008384,0.061676,0.031582,3
46,-0.169171,0.069617,-0.030523,0.020249,0.045904,0.146894,46
47,-0.175285,-0.200507,-0.083027,-0.080215,0.064870,0.141396,47
48,-0.016944,-0.042542,0.026981,-0.102993,-0.054991,0.059287,48
49,-0.121580,0.038042,-0.003330,0.079906,-0.080701,-0.067816,49


### ==========================================
### Structure K
### ==========================================

In [27]:
# ==========================================
# Structured MDN：Macro → pi，Micro + Industry → mu 和 sigma
# ==========================================
class StructuredMDN(nn.Module):
    def __init__(self, macro_dim, micro_dim, num_industries, ind_emb_dim=6, macro_hidden_dim=32,
                 micro_hidden_dim=64, head_dim=8, num_components=3, dropout_rate=0.05,
                 dist_type="normal"):
        super().__init__()
        self.num_components = num_components
        self.dist_type = dist_type

        self.ind_embedding = nn.Embedding(num_industries, ind_emb_dim)

        self.macro_net = nn.Sequential(
            nn.Linear(macro_dim, macro_hidden_dim),
            nn.LayerNorm(macro_hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(macro_hidden_dim, macro_hidden_dim // 2),
            nn.LayerNorm(macro_hidden_dim // 2),
            nn.ELU(),
        )

        self.micro_net = nn.Sequential(
            nn.Linear(micro_dim + ind_emb_dim, micro_hidden_dim),
            nn.LayerNorm(micro_hidden_dim),
            nn.ELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(micro_hidden_dim, micro_hidden_dim // 2),
            nn.LayerNorm(micro_hidden_dim // 2),
            nn.ELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(micro_hidden_dim // 2, head_dim),
            nn.LayerNorm(head_dim),
            nn.ELU(),
        )

        self.pi_head = nn.Linear(macro_hidden_dim // 2, num_components) if num_components > 1 else None
        self.mu_head = nn.Linear(head_dim, num_components)
        self.sigma_head = nn.Linear(head_dim, num_components)
        self.nu_head = nn.Linear(head_dim, num_components) if dist_type == "t" else None

        nn.init.normal_(self.ind_embedding.weight, mean=0.0, std=0.05)
        nn.init.xavier_normal_(self.mu_head.weight, gain=0.1)
        nn.init.zeros_(self.mu_head.bias)
        nn.init.xavier_normal_(self.sigma_head.weight, gain=0.1)
        nn.init.constant_(self.sigma_head.bias, -2.35)

        if self.pi_head is not None:
            nn.init.zeros_(self.pi_head.weight)
            nn.init.zeros_(self.pi_head.bias)

        if self.nu_head is not None:
            nn.init.xavier_normal_(self.nu_head.weight, gain=0.1)
            nn.init.constant_(self.nu_head.bias, 5.0)

    def forward(self, x_macro, x_micro, x_industry):
        h_macro = self.macro_net(x_macro)

        if self.pi_head is None:
            pi_logits = h_macro.new_zeros((len(h_macro), 1))
        else:
            pi_logits = self.pi_head(h_macro)

        industry_embedding = self.ind_embedding(x_industry)
        h_micro = self.micro_net(torch.cat([x_micro, industry_embedding], dim=1))

        mu = self.mu_head(h_micro)
        sigma = F.softplus(self.sigma_head(h_micro)) + sigma_floor
        nu = F.softplus(self.nu_head(h_micro)) + 2.01 if self.nu_head is not None else None

        return pi_logits, mu, sigma, nu

In [28]:
# ==========================================
# 运行 Structured MDN
# ==========================================
def run_structured_model(panel, num_components=3, dist_type="normal"):
    model_args = {
        "ind_emb_dim": 6,
        "macro_hidden_dim": 32,
        "micro_hidden_dim": 64,
        "head_dim": 8,
    }

    return run_mdn_model(
        panel, StructuredMDN, "structured", num_components, dist_type, model_args
    )

In [29]:
structured_forecasts, structured_metrics, structured_embeddings = run_structured_model(
    processed_df, num_components=3, dist_type="normal"
)

# 执行保存
structured_forecasts.to_parquet(os.path.join(path, f'structured_forecasts_{timestamp}.parquet'), index=False)
structured_metrics.to_parquet(os.path.join(path, f'structured_metrics_{timestamp}.parquet'), index=False)
structured_embeddings.to_parquet(os.path.join(path, f'structured_embeddings_{timestamp}.parquet'), index=False)

display(pd.concat([structured_forecasts.head(4), structured_forecasts.tail(4)]))
display(pd.concat([structured_metrics.head(2), structured_metrics.tail(2)]))
display(pd.concat([structured_embeddings.head(4), structured_embeddings.tail(4)]))

[structured K=3 | Window 0] Train: 1991-01~2005-12 | Val: 2006-01~2006-12 | Test: 2007-01~2007-12
Global seed set to 42 to ensure reproducibility.
[structured K=3 | Window 1] Train: 1992-01~2006-12 | Val: 2007-01~2007-12 | Test: 2008-01~2008-12
Global seed set to 43 to ensure reproducibility.


,permno,mthcaldt,target_ret_final,model_id,pi_vec,pi_logits_vec,mu_vec,sigma_vec,nu_vec
0,10001,2007-01-31,0.264774,structured_normal_k3,"[0.8806420564651489, 0.07940216362476349, 0.03...","[1.616028070449829, -0.7900973558425903, -1.47...","[0.011303320527076721, 0.11668604612350464, -0...","[0.0752384215593338, 0.23087932169437408, 0.10...",None
1,10002,2007-01-31,0.033054,structured_normal_k3,"[0.8806420564651489, 0.07940216362476349, 0.03...","[1.616028070449829, -0.7900973558425903, -1.47...","[0.030413813889026642, 0.11017809063196182, -0...","[0.07181519269943237, 0.14975281059741974, 0.0...",None
2,10025,2007-01-31,-0.028094,structured_normal_k3,"[0.8806420564651489, 0.07940216362476349, 0.03...","[1.616028070449829, -0.7900973558425903, -1.47...","[0.03312339633703232, 0.18368640542030334, -0....","[0.11911708116531372, 0.2601975202560425, 0.13...",None
3,10026,2007-01-31,-0.038517,structured_normal_k3,"[0.8806420564651489, 0.07940216362476349, 0.03...","[1.616028070449829, -0.7900973558425903, -1.47...","[0.026917044073343277, 0.1255684643983841, -0....","[0.07382248342037201, 0.17674332857131958, 0.0...",None
87371,92869,2008-12-31,-0.366006,structured_normal_k3,"[0.01035540271550417, 0.6265979409217834, 0.36...","[-2.7252583503723145, 1.3775380849838257, 0.83...","[-0.14619848132133484, -0.0001818239688873291,...","[0.12614786624908447, 0.1024029552936554, 0.37...",None
87372,92872,2008-12-31,-0.175000,structured_normal_k3,"[0.01035540271550417, 0.6265979409217834, 0.36...","[-2.7252583503723145, 1.3775380849838257, 0.83...","[-0.13095663487911224, 0.012489847838878632, 0...","[0.1305575966835022, 0.11417371779680252, 0.30...",None
87373,92874,2008-12-31,-0.496259,structured_normal_k3,"[0.01035540271550417, 0.6265979409217834, 0.36...","[-2.7252583503723145, 1.3775380849838257, 0.83...","[-0.05959434434771538, 0.013655390590429306, 0...","[0.08187542855739594, 0.06667669862508774, 0.2...",None
87374,93105,2008-12-31,0.106070,structured_normal_k3,"[0.01035540271550417, 0.6265979409217834, 0.36...","[-2.7252583503723145, 1.3775380849838257, 0.83...","[-0.15769922733306885, 0.011880731210112572, 0...","[0.14354285597801208, 0.120332270860672, 0.357...",None


,window,model_id,train_period,test_period,duration_sec,best_val_loss,epochs_run,train_loss_trace,val_loss_trace
0,0,structured_normal_k3,1991-01 to 2005-12,2007-01 to 2007-12,28.260019,-1.022142,2,"[-0.7614709634656776, -0.8009356968244217]","[-1.0221423261729152, -0.9928923018764871]"
1,1,structured_normal_k3,1992-01 to 2006-12,2008-01 to 2008-12,26.318926,-0.881331,2,"[-0.7664060615940601, -0.8195913453241234]","[-0.8744069024094687, -0.8813311771046132]"
0,0,structured_normal_k3,1991-01 to 2005-12,2007-01 to 2007-12,28.260019,-1.022142,2,"[-0.7614709634656776, -0.8009356968244217]","[-1.0221423261729152, -0.9928923018764871]"
1,1,structured_normal_k3,1992-01 to 2006-12,2008-01 to 2008-12,26.318926,-0.881331,2,"[-0.7664060615940601, -0.8195913453241234]","[-0.8744069024094687, -0.8813311771046132]"


,emb_dim_1,emb_dim_2,emb_dim_3,emb_dim_4,emb_dim_5,emb_dim_6,ffi49
0,-0.038895,-0.054896,0.173242,0.095757,-0.026849,-0.113339,0
1,0.183397,-0.215218,-0.029911,0.018591,-0.015938,-0.137051,1
2,0.043170,-0.177086,0.009153,0.078999,0.139749,-0.194125,2
3,-0.032425,-0.147853,0.029554,0.016176,0.145288,-0.016429,3
46,0.124917,-0.026603,-0.024544,-0.174653,0.001700,-0.100464,46
47,0.207706,-0.173706,0.117966,0.077421,0.050518,-0.156646,47
48,-0.032060,-0.013349,0.305104,0.067683,0.064789,-0.047946,48
49,0.068255,0.009917,-0.271414,-0.098707,-0.079705,0.029466,49


In [30]:
print(f"保存成功！文件位置: {path}")

保存成功！文件位置: /Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/output


In [31]:
if env == 'colab':
    # 运行完你的所有逻辑后，调用此命令会自动释放当前后端实例
    google.colab.runtime.unassign()